# MobileNetV2 - Phân loại biển báo giao thông (ver4 clean code)

- Dataset: file nén trên Google Drive, giải nén về `/content`, train bằng `SplitData/train/val/test`.
- Model: MobileNetV2 custom bằng PyTorch.
- Input: ảnh crop biển báo, mặc định 224x224.
- Mục tiêu ver4: giữ đủ pipeline train/evaluate/inference, bỏ phần lặp và code phụ quá dài.


In [ ]:
# CELL 1: Setup
import os, json, csv, math, random, shutil, subprocess, tarfile, time, warnings, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

from google.colab import drive
warnings.filterwarnings('ignore')
drive.mount('/content/drive')


In [ ]:
# CELL 2: Copy file nén từ Drive, giải nén về /content và tìm SplitData
DRIVE_ARCHIVE_PATH = '/content/drive/MyDrive/data_bien_bao.rar'
EXTRACT_ROOT = Path('/content/data_bien_bao')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

archive_path = Path(DRIVE_ARCHIVE_PATH)
if not archive_path.exists():
    raise FileNotFoundError(f'Không tìm thấy file nén: {archive_path}')

local_archive = Path('/content') / archive_path.name
if not local_archive.exists() or local_archive.stat().st_size != archive_path.stat().st_size:
    shutil.copy2(archive_path, local_archive)
print(f'Archive local: {local_archive} ({local_archive.stat().st_size / 1024**3:.2f} GB)')

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
suffix = ''.join(local_archive.suffixes).lower()
if suffix.endswith('.zip'):
    with zipfile.ZipFile(local_archive) as zf:
        zf.extractall(EXTRACT_ROOT)
elif suffix.endswith(('.tar', '.tar.gz', '.tgz')):
    with tarfile.open(local_archive, 'r:*') as tf:
        tf.extractall(EXTRACT_ROOT)
elif suffix.endswith('.rar'):
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=True)
    subprocess.run(['unrar', 'x', '-o+', str(local_archive), str(EXTRACT_ROOT) + '/'], check=True)
else:
    raise ValueError('Chỉ hỗ trợ .zip, .rar, .tar, .tar.gz, .tgz')

def count_images(folder):
    return sum(1 for p in Path(folder).rglob('*') if p.suffix.lower() in IMAGE_EXTS)

def is_splitdata(folder):
    folder = Path(folder)
    train = folder / 'train'
    return train.exists() and any(p.is_dir() and count_images(p) for p in train.iterdir())

split_candidates = [p for p in EXTRACT_ROOT.rglob('*') if p.is_dir() and p.name.lower() in {'splitdata', 'split_data', 'split-data'}]
DATA_DIR = next((p for p in split_candidates if is_splitdata(p)), None)
if DATA_DIR is None and is_splitdata(EXTRACT_ROOT):
    DATA_DIR = EXTRACT_ROOT
if DATA_DIR is None:
    raise RuntimeError('Không tìm thấy SplitData/train/<class>/*.jpg sau khi giải nén.')

print(f'DATA_DIR: {DATA_DIR}')
print(f'Tổng ảnh trong DATA_DIR: {count_images(DATA_DIR):,}')
print('Các thư mục cấp 1:', [p.name for p in sorted(DATA_DIR.iterdir()) if p.is_dir()])


In [ ]:
# CELL 3: Config
CONFIG = {
    'img_size': 224,
    'resize_enabled': 0,
    'resize_size': 224,
    'augment_enabled': 0,
    'batch_size': 32,
    'epochs': 50,
    'lr': 0.01,
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'width_mult': 1.0,
    'warmup_epochs': 5,
    'label_smoothing': 0.1,
    'dropout': 0.2,
    'grad_clip': 5.0,
    'patience': 10,
    'save_every': 10,
    'seed': 42,
    'resume': False,
    'checkpoint_dir': '/content/drive/MyDrive/mobilenetv2_gtsrb/checkpoints/',
    'log_dir': '/content/drive/MyDrive/mobilenetv2_gtsrb/logs/',
}

Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['log_dir']).mkdir(parents=True, exist_ok=True)

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# CELL 4: Dataset, normalize và DataLoader
DATA_DIR = Path(DATA_DIR)
IMG_SIZE = CONFIG['resize_size']
RESIZE_ENABLED = bool(CONFIG['resize_enabled'])
AUGMENT_ENABLED = bool(CONFIG['augment_enabled'])

def split_dir(name):
    for p in DATA_DIR.iterdir():
        if p.is_dir() and p.name.lower() == name:
            return p
    return None

train_dir = split_dir('train')
val_dir = split_dir('val') or split_dir('valid') or split_dir('validation')
test_dir = split_dir('test')
if not all([train_dir, val_dir, test_dir]):
    raise RuntimeError('Ver4 clean yêu cầu DATA_DIR có đủ train/val/test.')

def resize_steps(extra=0):
    return [transforms.Resize((IMG_SIZE + extra, IMG_SIZE + extra))] if RESIZE_ENABLED else []

def compute_mean_std(image_dir, max_images=3000):
    ds = ImageFolder(image_dir, transform=transforms.Compose(resize_steps() + [transforms.ToTensor()]))
    if len(ds) > max_images:
        g = torch.Generator().manual_seed(CONFIG['seed'])
        ds = Subset(ds, torch.randperm(len(ds), generator=g)[:max_images].tolist())
    loader = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
    total = torch.zeros(3)
    total_sq = torch.zeros(3)
    pixels = 0
    for images, _ in loader:
        total += images.sum(dim=(0, 2, 3))
        total_sq += (images ** 2).sum(dim=(0, 2, 3))
        pixels += images.numel() // 3
    mean = total / pixels
    std = torch.sqrt(total_sq / pixels - mean ** 2)
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(train_dir)

if AUGMENT_ENABLED:
    train_steps = resize_steps(extra=8) + [
        transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(20),
        transforms.RandomAffine(0, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
        transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
        transforms.RandomGrayscale(p=0.05),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
    ]
else:
    train_steps = resize_steps() + [transforms.ToTensor()]

augment_preview_transform = transforms.Compose(train_steps)
train_transform = transforms.Compose(train_steps + [transforms.Normalize(MEAN, STD)])
eval_transform = transforms.Compose(resize_steps() + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_dataset = ImageFolder(train_dir, transform=train_transform)
val_dataset = ImageFolder(val_dir, transform=eval_transform)
test_dataset = ImageFolder(test_dir, transform=eval_transform)
CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
CONFIG['num_classes'] = NUM_CLASSES

def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=shuffle, num_workers=2, pin_memory=(device.type == 'cuda'))

train_loader = make_loader(train_dataset, True)
val_loader = make_loader(val_dataset, False)
test_loader = make_loader(test_dataset, False)

print(f'Classes: {NUM_CLASSES} -> {CLASS_NAMES}')
print(f'Train/Val/Test: {len(train_dataset):,}/{len(val_dataset):,}/{len(test_dataset):,}')
print(f'Resize={int(RESIZE_ENABLED)}, Augment={int(AUGMENT_ENABLED)}, Batch={CONFIG["batch_size"]}')
print('Mean:', [round(x, 4) for x in MEAN])
print('Std: ', [round(x, 4) for x in STD])


In [ ]:
# CELL 5: Xem nhanh dữ liệu và augment mẫu
raw_preview = ImageFolder(train_dir, transform=None)
img, label = raw_preview[0]
to_pil = transforms.ToPILImage()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle(f'Augmentation example - {CLASS_NAMES[label]}', fontweight='bold')
axes = axes.ravel()
axes[0].imshow(img)
axes[0].set_title('Original')
axes[0].axis('off')
for i in range(1, len(axes)):
    axes[i].imshow(to_pil(augment_preview_transform(img)))
    axes[i].set_title(f'Augment {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

counts = Counter(label for _, label in raw_preview.samples)
plt.figure(figsize=(12, 4))
plt.bar([CLASS_NAMES[i] for i in range(NUM_CLASSES)], [counts[i] for i in range(NUM_CLASSES)])
plt.xticks(rotation=45, ha='right')
plt.title('Train images per class')
plt.tight_layout()
plt.show()


## MobileNetV2 custom

Inverted residual block: expand 1x1 -> depthwise 3x3 -> project 1x1. Skip connection dùng khi `stride=1` và số kênh không đổi.


In [ ]:
# ============================================================
# CELL 6: MOBILENETV2 - XÂY DỰNG TỪ ĐẦU
# ============================================================

def _make_divisible(v, divisor=8, min_value=None):
    """Đảm bảo channels chia hết cho divisor (tối ưu phần cứng)"""
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


class ConvBNReLU6(nn.Sequential):
    """Conv2d + BatchNorm2d + ReLU6"""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, groups=1):
        padding = (kernel_size - 1) // 2
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding,
                      groups=groups, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU6(inplace=True)
        )


class InvertedResidual(nn.Module):
    """
    Inverted Residual Block - Core building block của MobileNetV2

    Cấu trúc: Narrow → Wide → Narrow
    1. Expansion: 1×1 Conv (mở rộng channels) + BN + ReLU6
    2. Depthwise: 3×3 Conv (groups=channels) + BN + ReLU6
    3. Projection: 1×1 Conv (thu nhỏ channels) + BN (LINEAR - không ReLU!)

    Skip connection: chỉ khi stride=1 VÀ in_channels == out_channels
    """
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()
        self.stride = stride
        assert stride in [1, 2]

        hidden_dim = int(round(in_channels * expand_ratio))
        self.use_skip = (stride == 1 and in_channels == out_channels)

        layers = []
        # 1. Expansion layer (skip nếu expand_ratio = 1)
        if expand_ratio != 1:
            layers.append(ConvBNReLU6(in_channels, hidden_dim, kernel_size=1))

        # 2. Depthwise convolution
        layers.append(ConvBNReLU6(hidden_dim, hidden_dim, kernel_size=3,
                                   stride=stride, groups=hidden_dim))

        # 3. Projection layer (LINEAR - không có ReLU!)
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_channels),
        ])

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_skip:
            return x + self.conv(x)  # Skip connection
        else:
            return self.conv(x)


class MobileNetV2(nn.Module):
    """
    MobileNetV2 - Xây dựng từ đầu

    Architecture Table (theo paper gốc):
    | t | c   | n | s |
    |---|-----|---|---|
    | 1 | 16  | 1 | 1 |  (bottleneck)
    | 6 | 24  | 2 | 2 |
    | 6 | 32  | 3 | 2 |
    | 6 | 64  | 4 | 2 |
    | 6 | 96  | 3 | 1 |
    | 6 | 160 | 3 | 2 |
    | 6 | 320 | 1 | 1 |

    t = expansion factor, c = output channels
    n = repeat, s = stride (chỉ block đầu)
    """
    def __init__(self, num_classes=43, width_mult=1.0, dropout=0.2):
        super().__init__()

        # Cấu hình các stage: [expand_ratio, channels, num_blocks, stride]
        inverted_residual_setting = [
            # t, c,   n, s
            [1, 16,  1, 1],
            [6, 24,  2, 2],
            [6, 32,  3, 2],
            [6, 64,  4, 2],
            [6, 96,  3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]

        # === First layer: Conv2d 3×3 ===
        input_channels = _make_divisible(32 * width_mult)
        last_channels = _make_divisible(1280 * max(1.0, width_mult))

        first_stride = 2 if IMG_SIZE >= 160 else 1
        features = [ConvBNReLU6(3, input_channels, kernel_size=3, stride=first_stride)]
        # Use stride=2 for 224x224 input to reduce VRAM; keep stride=1 for smaller input.

        # === Inverted Residual Blocks ===
        for t, c, n, s in inverted_residual_setting:
            output_channels = _make_divisible(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(InvertedResidual(input_channels, output_channels,
                                                  stride=stride, expand_ratio=t))
                input_channels = output_channels

        # === Last conv layer: 1×1 ===
        features.append(ConvBNReLU6(input_channels, last_channels, kernel_size=1))

        self.features = nn.Sequential(*features)

        # === Classifier ===
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(last_channels, num_classes),
        )

        # === Weight Initialization (Kaiming) ===
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# Tạo model và kiểm tra
# Clear GPU memory before creating the model, useful after an OOM in the same runtime.
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = MobileNetV2(
    num_classes=CONFIG['num_classes'],
    width_mult=CONFIG['width_mult'],
    dropout=CONFIG['dropout']
).to(device)

# Test forward pass
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
out = model(dummy)
print(f"✅ Model tạo thành công!")
print(f"   Input:  {dummy.shape}")
print(f"   Output: {out.shape}")

# Đếm parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model Statistics:")
print(f"   Total params:     {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")
print(f"   Model size:       ~{total_params * 4 / 1024 / 1024:.1f} MB (FP32)")

# In kiến trúc
print(f"\n🏗️ Kiến trúc MobileNetV2:")
print(model)

In [ ]:
# CELL 7: Loss, optimizer và scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
optimizer = optim.SGD(
    model.parameters(),
    lr=CONFIG['lr'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
    nesterov=True,
)

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0

    def _lr(self):
        if self.current_epoch <= self.warmup_epochs:
            return self.base_lr * self.current_epoch / max(1, self.warmup_epochs)
        progress = (self.current_epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
        return self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))

    def step(self):
        self.current_epoch += 1
        lr = self._lr()
        for group in self.optimizer.param_groups:
            group['lr'] = lr
        return lr

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

    def state_dict(self):
        return {'current_epoch': self.current_epoch}

    def load_state_dict(self, state):
        self.current_epoch = state['current_epoch']

scheduler = WarmupCosineScheduler(optimizer, CONFIG['warmup_epochs'], CONFIG['epochs'], CONFIG['lr'])
print('Loss: CrossEntropyLoss | Optimizer: SGD Nesterov | Scheduler: Warmup + Cosine')


In [ ]:
# CELL 8: Training loop
from torch.cuda.amp import GradScaler, autocast

USE_AMP = device.type == 'cuda'

def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss = correct = total = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * labels.size(0)
            correct += outputs.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100 * correct / total

def save_ckpt(path, epoch, best_acc):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_acc': best_acc,
        'training_log': training_log,
        'class_names': CLASS_NAMES,
        'config': CONFIG,
    }, path)

def save_history():
    log_dir = Path(CONFIG['log_dir'])
    (log_dir / 'training_log.json').write_text(json.dumps(training_log, indent=2), encoding='utf-8')
    with open(log_dir / 'training_history.csv', 'w', newline='', encoding='utf-8') as f:
        keys = ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr', 'epoch_time']
        writer = csv.writer(f)
        writer.writerow(keys)
        writer.writerows(zip(*(training_log[k] for k in keys)))

training_log = {k: [] for k in ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr', 'epoch_time']}
start_epoch, best_val_acc, patience_counter = 0, 0, 0
latest_ckpt = Path(CONFIG['checkpoint_dir']) / 'checkpoint_latest.pth'

if CONFIG['resume'] and latest_ckpt.exists():
    ckpt = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    training_log = ckpt.get('training_log', training_log)
    start_epoch = ckpt.get('epoch', 0)
    best_val_acc = ckpt.get('best_val_acc', 0)
    print(f'Resume từ epoch {start_epoch}, best_val_acc={best_val_acc:.2f}%')

scaler = GradScaler(enabled=USE_AMP)
print('Epoch | Train Loss | Train Acc | Val Loss | Val Acc | LR | Time | Status')
for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
    t0 = time.time()
    lr = scheduler.step()
    train_loss, train_acc = run_epoch(model, train_loader, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, train=False)
    elapsed = time.time() - t0

    for key, value in zip(training_log, [epoch, train_loss, train_acc, val_loss, val_acc, lr, elapsed]):
        training_log[key].append(value)

    if val_acc > best_val_acc:
        best_val_acc, patience_counter, status = val_acc, 0, 'BEST'
        save_ckpt(Path(CONFIG['checkpoint_dir']) / 'best_model.pth', epoch, best_val_acc)
    else:
        patience_counter += 1
        status = f'{patience_counter}/{CONFIG["patience"]}'

    save_ckpt(latest_ckpt, epoch, best_val_acc)
    if epoch % CONFIG['save_every'] == 0:
        save_ckpt(Path(CONFIG['checkpoint_dir']) / f'checkpoint_epoch_{epoch}.pth', epoch, best_val_acc)
    save_history()

    print(f'{epoch:5d} | {train_loss:10.4f} | {train_acc:8.2f}% | {val_loss:8.4f} | {val_acc:7.2f}% | {lr:.6f} | {elapsed:5.1f}s | {status}')
    if patience_counter >= CONFIG['patience']:
        print(f'Early stopping tại epoch {epoch}')
        break

print(f'Best val accuracy: {best_val_acc:.2f}%')


In [ ]:
# CELL 9: Vẽ lịch sử training
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs = training_log['epoch']
axes[0].plot(epochs, training_log['train_loss'], label='train')
axes[0].plot(epochs, training_log['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, training_log['train_acc'], label='train')
axes[1].plot(epochs, training_log['val_acc'], label='val')
axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# CELL 10: Đánh giá test set
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

best_ckpt = Path(CONFIG['checkpoint_dir']) / 'best_model.pth'
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model: val_acc={ckpt.get("best_val_acc", 0):.2f}%')

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc = accuracy_score(all_labels, all_preds) * 100
test_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0) * 100
test_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0) * 100
test_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0) * 100

print(f'Test accuracy : {test_acc:.2f}%')
print(f'Precision     : {test_precision:.2f}%')
print(f'Recall        : {test_recall:.2f}%')
print(f'F1-score      : {test_f1:.2f}%')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# CELL 11: Confusion matrix và per-class accuracy
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

per_class_acc = cm.diagonal() / np.maximum(cm.sum(axis=1), 1) * 100
order = np.argsort(per_class_acc)
print('Worst classes:')
for i in order[:5]:
    print(f'{i:2d} {CLASS_NAMES[i]:30s}: {per_class_acc[i]:6.2f}%')
print('Best classes:')
for i in order[-5:][::-1]:
    print(f'{i:2d} {CLASS_NAMES[i]:30s}: {per_class_acc[i]:6.2f}%')


In [ ]:
# CELL 12: Hiển thị vài ảnh dự đoán đúng/sai
def unnormalize(tensor):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

def show_samples(indices, title, color):
    if len(indices) == 0:
        print(f'Không có mẫu cho nhóm: {title}')
        return
    chosen = np.linspace(0, len(indices) - 1, min(8, len(indices)), dtype=int)
    fig, axes = plt.subplots(1, len(chosen), figsize=(2.5 * len(chosen), 3))
    axes = np.atleast_1d(axes)
    for ax, pos in zip(axes, chosen):
        idx = int(indices[pos])
        img, label = test_dataset[idx]
        pred = all_preds[idx]
        ax.imshow(unnormalize(img).permute(1, 2, 0))
        ax.set_title(f'T:{CLASS_NAMES[label]}\nP:{CLASS_NAMES[pred]}', color=color, fontsize=9)
        ax.axis('off')
    fig.suptitle(title, color=color, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_samples(np.where(all_preds == all_labels)[0], 'Dự đoán đúng', 'green')
show_samples(np.where(all_preds != all_labels)[0], 'Dự đoán sai', 'red')


In [ ]:
# CELL 13: Model analysis và export
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = total_params * 4 / 1024**2

print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Model size      : {model_size_mb:.2f} MB')
print(f'Best val acc    : {best_val_acc:.2f}%')
print(f'Test acc        : {test_acc:.2f}%')

export_path = Path(CONFIG['checkpoint_dir']) / 'mobilenetv2_final.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'config': CONFIG,
    'mean': MEAN,
    'std': STD,
    'test_acc': test_acc,
}, export_path)
print('Exported:', export_path)


In [ ]:
# CELL 14: Dự đoán ảnh bên ngoài
from google.colab import files

# Nếu ảnh chưa crop, nhập tọa độ crop dạng (left, top, right, bottom). Để None nếu ảnh đã crop sẵn.
CROP_BOX = None

predict_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def predict_image(path, crop_box=None, topk=5):
    original = Image.open(path).convert('RGB')
    image = original.crop(crop_box) if crop_box else original
    x = predict_transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    values, indices = probs.topk(topk)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image)
    axes[0].set_title('Input crop' if crop_box else 'Input image')
    axes[0].axis('off')
    axes[1].barh([CLASS_NAMES[i] for i in indices.cpu().tolist()][::-1], (values.cpu().numpy() * 100)[::-1])
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('Top predictions')
    plt.tight_layout()
    plt.show()

    for p, i in zip(values.cpu().numpy() * 100, indices.cpu().tolist()):
        print(f'{CLASS_NAMES[i]:30s}: {p:6.2f}%')

uploaded = files.upload()
for fname, content in uploaded.items():
    path = Path('/content') / fname
    path.write_bytes(content)
    print('\nẢnh:', fname)
    predict_image(path, CROP_BOX)
